# Building Knowledge Stores

Create a knowledge store and add videos to build a persistent, queryable collection of videos plus derived understanding. This notebook covers creating stores with different ingestion configurations and adding assets.

In [ ]:
import json
import os
import time

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

## When You Need This

You have assets uploaded and want to organize them into a collection that Jockey can reason over. A knowledge store contains your videos plus derived understanding: spatiotemporal context, a typed ontology, and embeddings that enable semantic retrieval and corpus-level reasoning.

## Helper: Wait for Item Indexing

In [ ]:
def wait_for_item_ready(
    store_id: str,
    item_id: str,
    headers: dict,
    base_url: str = BASE_URL,
    interval: int = 10,
    timeout: int = 600,
) -> dict:
    """Poll a knowledge store item until it reaches 'ready' or 'failed' status.

    Args:
        store_id: The knowledge store ID.
        item_id: The item ID to monitor.
        headers: Request headers including the API key.
        base_url: The base URL for the Jockey API.
        interval: Seconds between polling attempts.
        timeout: Maximum seconds to wait before raising an error.

    Returns:
        The item response dict once it reaches 'ready' status.

    Raises:
        Exception: If the item fails indexing or the timeout is exceeded.
    """
    elapsed = 0
    while elapsed < timeout:
        response = requests.get(
            f"{base_url}/knowledge-stores/{store_id}/items/{item_id}",
            headers=headers,
        )
        item_data = response.json()
        status = item_data["status"]

        if status == "ready":
            print(f"Item {item_id} is ready.")
            return item_data
        elif status == "failed":
            raise Exception(f"Item {item_id} indexing failed.")

        print(f"Item status: {status} (elapsed: {elapsed}s)")
        time.sleep(interval)
        elapsed += interval

    raise Exception(f"Timeout after {timeout}s waiting for item {item_id}.")

## Create a Basic Knowledge Store

The simplest knowledge store requires only a name. Jockey will use default extraction settings.

In [ ]:
response = requests.post(
    f"{BASE_URL}/knowledge-stores",
    headers=HEADERS,
    json={"name": "My Video Collection"},
)

store = response.json()
STORE_ID = store["_id"]
print(f"Knowledge Store ID: {STORE_ID}")
print(json.dumps(store, indent=2))

## Create with Ingestion Config: Natural Language

Shape what Jockey extracts by providing a natural language description. Jockey converts it to a schema internally.

In [ ]:
response = requests.post(
    f"{BASE_URL}/knowledge-stores",
    headers=HEADERS,
    json={
        "name": "Marketing Analysis",
        "ingestion_config": {
            "enrichment_config": {
                "type": "description",
                "description": (
                    "Focus on brand mentions, product appearances, "
                    "audience reactions, and visual tone"
                )
            }
        },
    },
)

marketing_store = response.json()
print(f"Marketing Store ID: {marketing_store['_id']}")

## Create with Ingestion Config: JSON Schema

For precise, structured extraction, provide a JSON Schema to define the exact fields you need.

In [ ]:
response = requests.post(
    f"{BASE_URL}/knowledge-stores",
    headers=HEADERS,
    json={
        "name": "Security Monitoring",
        "ingestion_config": {
            "enrichment_config": {
                "type": "json_schema",
                "json_schema": {
                    "type": "object",
                    "properties": {
                        "people_count": {"type": "integer"},
                        "location": {"type": "string"},
                        "suspicious_activity": {"type": "boolean"},
                        "description": {"type": "string"},
                    },
                }
            }
        },
    },
)

security_store = response.json()
print(f"Security Store ID: {security_store['_id']}")

## Add a Video to the Store

Once you have a knowledge store and a `ready` asset, add the asset to the store. Indexing happens asynchronously.

In [ ]:
ASSET_ID = "your_asset_id"  # Replace with an actual asset ID

response = requests.post(
    f"{BASE_URL}/knowledge-stores/{STORE_ID}/items",
    headers=HEADERS,
    json={"asset_id": ASSET_ID},
)

item = response.json()
item_id = item["_id"]
print(f"Item ID: {item_id}")

# Wait for indexing to complete
ready_item = wait_for_item_ready(STORE_ID, item_id, HEADERS)
print("Video indexed and ready.")

## Add Multiple Videos

Add a batch of assets and wait for all to finish indexing.

In [ ]:
ASSET_IDS = ["asset_1", "asset_2", "asset_3"]  # Replace with actual asset IDs
item_ids: list[str] = []

# Add all videos
for asset_id in ASSET_IDS:
    response = requests.post(
        f"{BASE_URL}/knowledge-stores/{STORE_ID}/items",
        headers=HEADERS,
        json={"asset_id": asset_id},
    )
    item_ids.append(response.json()["_id"])
    print(f"Added asset {asset_id} -> item {item_ids[-1]}")

# Wait for all items to be ready
for item_id in item_ids:
    while True:
        response = requests.get(
            f"{BASE_URL}/knowledge-stores/{STORE_ID}/items/{item_id}",
            headers=HEADERS,
        )
        status = response.json()["status"]
        if status == "ready":
            print(f"Item {item_id} is ready.")
            break
        elif status == "failed":
            print(f"Item {item_id} failed.")
            break
        time.sleep(10)

print(f"All {len(item_ids)} videos processed.")

## Common Pitfalls

- **Asset must be ready first.** Adding a `processing` asset will fail. Wait for the asset to reach `ready` status.
- **Indexing takes time.** Expect 1-10 minutes per video depending on length.

## Next Steps

- [Ingestion Config](ingestion_config.ipynb) — learn more about configuring extraction
- [Querying](querying.ipynb) — ask questions about your indexed video collection
- [Authentication](authentication.ipynb) — API key setup and security

**API Reference:**
- [POST /knowledge-stores](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/knowledge-stores/create-knowledge-store)
- [POST /knowledge-stores/{id}/items](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/knowledge-store-items/create-knowledge-store-item)